# Orbital Performance Report

Enable **Write performance CSV** in the debug UI's Performance panel, then copy the resulting file here or set `ORBITAL_PERF_CSV` to its path.

The notebook computes summary statistics and automated observations, then plots FPS, frame time, distributions, CPU/GPU stages, and cache counters.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path(os.environ.get("ORBITAL_PERF_CSV", "Logs/performance.csv")).expanduser()
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Copy performance.csv here or set ORBITAL_PERF_CSV."
    )

df = pd.read_csv(CSV_PATH).sort_values("elapsed_s").reset_index(drop=True)
df.head()

In [ ]:
CACHE_COUNTERS = [
    "shader_modules_delta",
    "pipelines_delta",
    "material_hits_delta",
    "material_misses_delta",
]
SKY_COUNTER = "sky_updates_delta"
COUNTERS = [*CACHE_COUNTERS, "unique_wgsl", SKY_COUNTER]
STAGES = [c for c in df.columns if c not in {"elapsed_s", "fps", "cdt_ms", *COUNTERS}]
CPU_STAGES = [c for c in STAGES if not c.startswith("gpu_") and c != "total"]
GPU_STAGES = [c for c in STAGES if c.startswith("gpu_")]
TOTAL = df["total"] if "total" in df else df["cdt_ms"]

print(f"{len(df)} samples over {df['elapsed_s'].max() - df['elapsed_s'].min():.1f}s")
print(f"CPU stages: {', '.join(CPU_STAGES)}")
print(f"GPU stages: {', '.join(GPU_STAGES) if GPU_STAGES else 'none'}")

## Summary

`fps` is the one-second frame count. `total` is the measured frame duration; `cdt_ms` is the wall-clock delta between frames.

In [ ]:
def pct(series, q):
    return float(np.nanpercentile(series.astype(float), q))

fps = df["fps"].astype(float)
cdt = df["cdt_ms"].astype(float)
means = df[STAGES].mean(numeric_only=True).sort_values(ascending=False)

summary = pd.Series({
    "samples": len(df),
    "duration_s": float(df["elapsed_s"].max() - df["elapsed_s"].min()),
    "fps_mean": float(fps.mean()),
    "fps_median": float(fps.median()),
    "fps_p95": pct(fps, 95),
    "fps_p99": pct(fps, 99),
    "frame_total_mean_ms": float(TOTAL.mean()),
    "frame_total_p95_ms": pct(TOTAL, 95),
    "frame_total_p99_ms": pct(TOTAL, 99),
    "cdt_mean_ms": float(cdt.mean()),
    "cdt_p95_ms": pct(cdt, 95),
    "slowest_stage": str(means.index[0]) if len(means) else "-",
    "slowest_stage_mean_ms": float(means.iloc[0]) if len(means) else 0.0,
})
summary.to_frame("value")

In [ ]:
spike_threshold = max(float(TOTAL.median()) * 2.0, 1e-6)
spikes = int((TOTAL > spike_threshold).sum())
fps_cv = float(fps.std() / fps.mean() * 100.0) if fps.mean() else 0.0

print(f"Frame-time spikes above {spike_threshold:.2f} ms: {spikes}")
print(f"FPS coefficient of variation: {fps_cv:.1f}%")
print("\nMean stage cost:")
for name, value in means.head(12).items():
    share = value / TOTAL.mean() * 100.0 if TOTAL.mean() else 0.0
    print(f"  {name:24s} {value:8.3f} ms  ({share:5.1f}%)")

if fps_cv > 10.0:
    print("\nFPS is noticeably variable; inspect the frame-time and stage graphs.")
if spikes:
    print("Spikes are present; compare their timestamps with stage peaks.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df["elapsed_s"], fps, color="#4aa3ff", linewidth=1.2)
axes[0].set_title("FPS")
axes[0].set_ylabel("frames/s")
axes[0].grid(alpha=0.25)

axes[1].plot(df["elapsed_s"], TOTAL, label="frame total", color="#ffb347", linewidth=1.1)
axes[1].plot(df["elapsed_s"], cdt, label="CDT", color="#d16bff", linewidth=0.9)
axes[1].set_title("Frame time")
axes[1].set_ylabel("ms")
axes[1].set_xlabel("elapsed_s")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(fps, bins=30, color="#4aa3ff", alpha=0.8)
axes[0].axvline(fps.median(), color="#ffb347", linestyle="--", label="median")
axes[0].set_title("FPS distribution")
axes[0].set_xlabel("frames/s")
axes[0].legend()

axes[1].hist(TOTAL, bins=40, color="#ff8c69", alpha=0.8)
axes[1].axvline(TOTAL.median(), color="#ffb347", linestyle="--", label="median")
axes[1].set_title("Frame-time distribution")
axes[1].set_xlabel("ms")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
if CPU_STAGES:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    cpu_means = df[CPU_STAGES].mean(numeric_only=True).sort_values()
    axes[0].barh(cpu_means.index, cpu_means.values, color="#55c98a")
    axes[0].set_title("CPU stage mean")
    axes[0].set_xlabel("ms")

    top = cpu_means.sort_values(ascending=False).head(6).index
    for name in top:
        axes[1].plot(df["elapsed_s"], df[name], label=name, linewidth=0.9)
    axes[1].set_title("Top CPU stages over time")
    axes[1].set_xlabel("elapsed_s")
    axes[1].set_ylabel("ms")
    axes[1].legend(fontsize=7, ncol=2)
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("No CPU stage columns found.")

In [ ]:
if CPU_STAGES:
    top_cpu = df[CPU_STAGES].mean(numeric_only=True).sort_values(ascending=False).head(10).index
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].stackplot(
        df["elapsed_s"],
        *[df[name] for name in top_cpu],
        labels=top_cpu,
        alpha=0.85,
    )
    axes[0].set_title("CPU composition (top 10 stages)")
    axes[0].set_xlabel("elapsed_s")
    axes[0].set_ylabel("ms")
    cpu_means = df[CPU_STAGES].mean(numeric_only=True)
    shares = (cpu_means.loc[top_cpu] / TOTAL.mean() * 100.0).sort_values()
    axes[1].barh(shares.index, shares.values, color="#79d6a6")
    axes[1].set_title("CPU share of mean frame time")
    axes[1].set_xlabel("%")
    plt.tight_layout()
    plt.show()
else:
    print("No CPU stage columns found.")

In [ ]:
if GPU_STAGES:
    gpu_means = df[GPU_STAGES].mean(numeric_only=True).sort_values()
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for name in GPU_STAGES:
        axes[0].plot(df["elapsed_s"], df[name], label=name, linewidth=0.9)
    axes[0].set_title("GPU stages over time")
    axes[0].set_xlabel("elapsed_s")
    axes[0].set_ylabel("ms")
    axes[0].legend(fontsize=7, ncol=2)
    axes[0].grid(alpha=0.25)

    axes[1].barh(gpu_means.index, gpu_means.values, color="#ff8c69")
    axes[1].set_title("GPU stage mean")
    axes[1].set_xlabel("ms")
    plt.tight_layout()
    plt.show()
else:
    print("No GPU stage columns found; timestamp queries may be unavailable.")

In [ ]:
rolling_window = max(3, min(15, len(TOTAL) // 20 or 3))
rolling = TOTAL.rolling(window=rolling_window, min_periods=1).mean()
spike_mask = TOTAL > spike_threshold

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df["elapsed_s"], TOTAL, label="frame total", color="#ffb347", alpha=0.5, linewidth=0.9)
ax.plot(df["elapsed_s"], rolling, label=f"rolling mean ({rolling_window})", color="#4aa3ff", linewidth=1.5)
ax.scatter(
    df.loc[spike_mask, "elapsed_s"],
    TOTAL[spike_mask],
    color="#ff4f4f",
    s=18,
    label=f"spikes ({spikes})",
    zorder=3,
)
ax.axhline(TOTAL.median(), color="#55c98a", linestyle="--", linewidth=1, label="median")
ax.set_title("Frame-time spikes and rolling mean")
ax.set_xlabel("elapsed_s")
ax.set_ylabel("ms")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
cache_present = [name for name in CACHE_COUNTERS if name in df]
if cache_present:
    for name in cache_present:
        axes[0].plot(df["elapsed_s"], df[name], label=name, linewidth=0.9)
    axes[0].set_title("Cache activity (per window)")
    axes[0].legend(fontsize=7, ncol=2)
else:
    axes[0].set_visible(False)

if "unique_wgsl" in df:
    axes[1].plot(df["elapsed_s"], df["unique_wgsl"], color="#4aa3ff")
    axes[1].set_title("Unique WGSL sources (cumulative)")
else:
    axes[1].set_visible(False)

if SKY_COUNTER in df:
    axes[2].plot(df["elapsed_s"], df[SKY_COUNTER], color="#ffb347")
    axes[2].set_title("Dynamic sky updates (per window)")
    axes[2].set_ylabel("updates")
else:
    axes[2].set_visible(False)

for ax in axes:
    if ax.get_visible():
        ax.set_xlabel("elapsed_s")
        ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Reading the result

- A high `total` with a low `present` usually means CPU work before submission.
- `scene_encode`, `scene_submit`, `cull+extract`, or `bind+models` growing together points at scene complexity or CPU-side preparation.
- `gpu_*` growth points at the GPU pass; if they are all zero, timestamp queries are unavailable on that backend.
- `gpu_volumetrics`, `gpu_tonemap`, and `gpu_aa` cover post-processing GPU work; `gpu_scene_total` covers shadow/skybox/models only.
- `realize` and `realize_models` are usually one-time setup costs, but if they stay high they indicate models are being treated as dirty every frame.
- `surface_acq` is the total, split into `surface_lock`, `surface_acquire`, and `surface_view`. A high `surface_acquire` is often swapchain/GPU backpressure, not CPU computation.
- `material_misses_delta` or `pipelines_delta` growing after startup often means materials or shaders are being created at runtime.
- `sky_updates_delta` is the dynamic-sky update count, not a cache metric. A dynamic sky can legitimately update roughly once per frame; compare it with FPS rather than the cache counters.
- `unique_wgsl` is cumulative. The other cache counters are per-window deltas, so they are plotted on separate axes.
- Compare the mean and p99, not just the average: a good mean with a bad p99 usually means hitching or occasional compile/upload work.